<a href="https://colab.research.google.com/github/rohini-th/Ai/blob/main/Final_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import nltk
from nltk.tokenize import word_tokenize

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, LSTM, Embedding, Dropout
from tensorflow.keras.callbacks import EarlyStopping

In [2]:

df = pd.read_csv("/content/Reviews.csv", usecols=["Text"])

df = df.head(3000)

df.head()

,Text
0,I have bought several of the Vitality canned d...
1,Product arrived labeled as Jumbo Salted Peanut...
2,This is a confection that has been around a fe...
3,If you are looking for the secret ingredient i...
4,Great taffy at a great price. There was a wid...


In [3]:
import re
def clean_text(text):

    text = str(text).lower()

    text = re.sub(r'[^a-zA-Z ]','',text)

    text = re.sub(r'\s+',' ',text)

    return text

df["Text"] = df["Text"].apply(clean_text)

In [4]:
df = df[df["Text"]!=""]

**Tokenization**

In [5]:
tokenizer = Tokenizer(num_words=5000, oov_token="<OOV>")
tokenizer.fit_on_texts(df["Text"])

total_words = 5000
print(total_words)

5000


In [6]:
print(len(tokenizer.word_index)+1)

12148


In [7]:
wordindex = tokenizer.word_index
reverse_word_index = {index:word for word,index in wordindex.items()}
print(reverse_word_index)

{1: '<OOV>', 2: 'the', 3: 'i', 4: 'and', 5: 'a', 6: 'to', 7: 'of', 8: 'it', 9: 'is', 10: 'this', 11: 'for', 12: 'in', 13: 'my', 14: 'are', 15: 'that', 16: 'but', 17: 'with', 18: 'you', 19: 'not', 20: 'they', 21: 'have', 22: 'was', 23: 'these', 24: 'as', 25: 'br', 26: 'on', 27: 'so', 28: 'like', 29: 'them', 30: 'good', 31: 'chips', 32: 'or', 33: 'be', 34: 'great', 35: 'at', 36: 'just', 37: 'very', 38: 'if', 39: 'all', 40: 'taste', 41: 'flavor', 42: 'its', 43: 'one', 44: 'product', 45: 'we', 46: 'from', 47: 'can', 48: 'love', 49: 'tea', 50: 'had', 51: 'when', 52: 'than', 53: 'will', 54: 'more', 55: 'were', 56: 'me', 57: 'has', 58: 'other', 59: 'food', 60: 'out', 61: 'would', 62: 'really', 63: 'some', 64: 'no', 65: 'only', 66: 'about', 67: 'get', 68: 'too', 69: 'much', 70: 'dont', 71: 'because', 72: 'bag', 73: 'also', 74: 'an', 75: 'your', 76: 'best', 77: 'buy', 78: 'am', 79: 'time', 80: 'better', 81: 'find', 82: 'up', 83: 'coffee', 84: 'use', 85: 'little', 86: 'there', 87: 'amazon', 88: 

**Generate the Input sequences and then apply pad sequences**

In [8]:
df[:10]

,Text
0,i have bought several of the vitality canned d...
1,product arrived labeled as jumbo salted peanut...
2,this is a confection that has been around a fe...
3,if you are looking for the secret ingredient i...
4,great taffy at a great price there was a wide ...
5,i got a wild hair for taffy and ordered this f...
6,this saltwater taffy had great flavors and was...
7,this taffy is so good it is very soft and chew...
8,right now im mostly just sprouting this so my ...
9,this is a very healthy dog food good for their...


In [9]:
input_sequences=[]

for line in df["Text"]:

    token_list = tokenizer.texts_to_sequences([line])[0]

    for i in range(1,len(token_list)):

        n_gram_sequence = token_list[:i+1]

        input_sequences.append(n_gram_sequence)
print(input_sequences[:5])

[[3, 21], [3, 21, 122], [3, 21, 122, 333], [3, 21, 122, 333, 7], [3, 21, 122, 333, 7, 2]]


**Paddingt**

In [10]:
MAX_LEN = 30

input_sequences = pad_sequences(
    input_sequences,
    maxlen=MAX_LEN,
    padding='pre',
    truncating='pre'
)

In [11]:
X = input_sequences[:, :-1]
y = input_sequences[:, -1].astype("int32")
print(y.shape)

(220096,)


In [12]:
model = Sequential()

model.add(
    Embedding(
        input_dim=total_words,
        output_dim=64
    )
)

model.add(LSTM(64))
model.add(Dense(total_words, activation="softmax"))

In [13]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [14]:
import tensorflow as tf

print("Total Words:", total_words)

model.summary()

print("X dtype:", X.dtype)
print("y dtype:", y.dtype)

print("GPU:", tf.config.list_physical_devices('GPU'))

Total Words: 5000


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

X dtype: int32
y dtype: int32
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [28]:
X_small = X[:50000]
y_small = y[:50000]

from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="loss",
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    X_small,
    y_small,
    epochs=20,
    batch_size=32,
    callbacks=[early_stop],
    validation_split=0.1
)

Epoch 1/20
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - accuracy: 0.4348 - loss: 2.6443 - val_accuracy: 0.4140 - val_loss: 2.7220
Epoch 2/20
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 11s 8ms/step - accuracy: 0.4547 - loss: 2.5388 - val_accuracy: 0.3830 - val_loss: 2.8547
Epoch 3/20
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 11s 8ms/step - accuracy: 0.4722 - loss: 2.4532 - val_accuracy: 0.3658 - val_loss: 2.9861
Epoch 4/20
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 11s 8ms/step - accuracy: 0.4887 - loss: 2.3591 - val_accuracy: 0.3400 - val_loss: 3.1129
Epoch 5/20
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - accuracy: 0.5085 - loss: 2.2698 - val_accuracy: 0.3220 - val_loss: 3.2479
Epoch 6/20
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 11s 8ms/step - accuracy: 0.5238 - loss: 2.1849 - val_accuracy: 0.3068 - val_loss: 3.3750
Epoch 7/20
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - accuracy: 0.5390 - loss: 2.1087 - val_accuracy: 0.2914 - val_loss: 3.5133
Epoch 8/20
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 20s 8ms/step - accuracy: 0.5556 - loss: 2

In [26]:

def generate_text(seed_text, next_words):

    for _ in range(next_words):

        token_list = tokenizer.texts_to_sequences([seed_text])[0]

        token_list = pad_sequences(
            [token_list],
            maxlen=MAX_LEN - 1,
            padding='pre',
            truncating='pre'
        )

        preds = model.predict(token_list, verbose=0)[0]
        temperature = 0.8
        preds = np.log(preds + 1e-10) / temperature
        preds = np.exp(preds)
        preds = preds / np.sum(preds)
        predicted = np.random.choice(len(preds), p=preds)
        output_word = tokenizer.index_word.get(predicted, "")

        # Stop if no word is found
        if output_word == "":
            break

        seed_text += " " + output_word

    return seed_text

In [27]:
print(generate_text("this food",20))

print(generate_text("i love",50))

print(generate_text("the taste",20))

this food was fast and simply almost buy weight carries this product again and different food the sugar version my family loves
i love this for my year of my pet food i use this product again but we bought this order i tried it in the flavors wow i will reorder with the past years they are the best i like ever eaten they are not wonderfully crunchy and the only thing were
the taste of light and crispy extremely nice tasty well crunchy chips i love them they are not too too rich or
